# FYOE v3 — train the production model on Colab T4

**Before you start:** click `Runtime` → `Change runtime type` → set hardware accelerator to **T4 GPU**.

Then `Runtime` → `Run all`. ~30 min total. At the end you'll get a `saved_model.zip` download — drop the unzipped folder into your backend repo at `./saved_model/`, restart the API, done.

**What this notebook does:**
1. Clones the `v3` branch.
2. Generates ~50K training examples (positives + hard negatives).
3. Fine-tunes `microsoft/deberta-v3-base` for 18 intents with class-balanced loss.
4. Tunes per-intent thresholds on the val set (saved as `thresholds.json`).
5. Reports per-intent precision/recall/F1.
6. Zips `saved_model/` for download.

## 1. Setup — clone the repo and install dependencies

In [ ]:
!nvidia-smi  # confirm T4 is attached

In [ ]:
!git clone -b v3 https://github.com/Akash-Cheerla/paychat-model.git
%cd paychat-model
!git log -1 --oneline

In [ ]:
# DeBERTa-v3 needs sentencepiece for its tokenizer; the rest are usually preinstalled on Colab.
!pip install -q sentencepiece transformers scikit-learn
import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)

## 2. Generate training data

~50K examples: positives across 18 intents, multi-intent mixes, generic chitchat, and per-intent hard negatives ("I love Paris" must NOT fire travel).

In [ ]:
%cd training
!python generate_data.py

## 3. Fine-tune DeBERTa-v3-base

5 epochs, batch 32. The trainer auto-picks LR by model architecture (5e-6 for DeBERTa-v3, 2e-5 for BERT-family) — DeBERTa diverges to NaN at higher LRs. ~25-30 min on T4. Class imbalance is handled via `pos_weight` and per-intent thresholds are tuned on the val set after the final epoch.

In [ ]:
!python train.py \
  --model microsoft/deberta-v3-base \
  --epochs 5 \
  --batch-size 32 \
  --max-len 128

## 4. Inspect results

In [ ]:
import json
from pathlib import Path
model_dir = Path('../saved_model')
print('Files in saved_model/:')
for f in sorted(model_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:35} {size_kb:>10.1f} KB')

print('\nLearned per-intent thresholds:')
with open(model_dir / 'thresholds.json') as f:
    thresholds = json.load(f)
for intent, thr in thresholds.items():
    print(f'  {intent:<14} {thr:.2f}')

print('\nPer-intent test metrics:')
with open(model_dir / 'training_report.json') as f:
    report = json.load(f)
print(f"  test exact-match: {report['test_exact_match']:.2%}")
print(f"  test hamming:     {report['test_hamming']:.2%}")
print()
print(f"  {'intent':<14} {'precision':>10} {'recall':>8} {'f1':>7}")
for intent, m in report['per_intent'].items():
    print(f"  {intent:<14} {m['precision']:>9.1%} {m['recall']:>7.1%} {m['f1']:>6.1%}")

## 5. Quick sanity check — fire a few real messages through the model

In [ ]:
import torch, json
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tok = AutoTokenizer.from_pretrained('../saved_model')
mdl = AutoModelForSequenceClassification.from_pretrained('../saved_model').eval().cuda()
with open('../saved_model/thresholds.json') as f:
    thresholds = json.load(f)
labels = list(mdl.config.id2label.values()) if mdl.config.id2label else list(thresholds.keys())

samples = [
    'venmo me 20 bucks for pizza',
    'remind me to call mom tomorrow at 6pm',
    "let's get sushi tonight, doordash it",
    'uber to JFK at 5am tomorrow',
    'flights to Tokyo next month',
    'play Anti-Hero by Taylor Swift',
    'movie night, watching Oppenheimer',
    "book a table at Blue Bottle for 4 people friday at 7",
    'rent due friday, transfer 1200',
    "is it gonna rain in mumbai tomorrow",
    'I love Paris',                        # should NOT fire travel
    'watched Stranger Things last week',   # should NOT fire video
]

for text in samples:
    enc = tok(text, return_tensors='pt', truncation=True, max_length=128).to('cuda')
    with torch.no_grad():
        logits = mdl(**enc).logits[0]
    probs = torch.sigmoid(logits).cpu().tolist()
    fired = [(labels[i], probs[i]) for i in range(len(labels)) if probs[i] >= thresholds.get(labels[i], 0.5)]
    fired.sort(key=lambda x: -x[1])
    fired_str = ', '.join(f'{l}={p:.2f}' for l, p in fired) or '(none)'
    print(f'{text!r:<60} -> {fired_str}')

## 6. Zip and download `saved_model/`

Drop the unzipped folder into your backend repo at `./saved_model/` (replacing the v2 weights), commit it (or store on S3/HF Hub for production), and restart the FastAPI server. iOS and Android keep calling `POST /detect` unchanged.

In [ ]:
%cd ..
!zip -r saved_model_v3.zip saved_model/ -x '*.bin.tmp' '*.cache*' > /dev/null
!ls -lh saved_model_v3.zip
from google.colab import files
files.download('saved_model_v3.zip')